<a href="https://colab.research.google.com/github/shikhar-iimc/Prajna-previous_Versions/blob/main/Prajna_Demo_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# ============================================
# CELL 1 — Install packages (UPDATED)
# ============================================
!pip install requests pandas neo4j groq spacy pyvis streamlit trafilatura -q
!pip install newspaper3k lxml_html_clean -q
!python -m spacy download en_core_web_sm -q

print("✅ Packages installed!")

import requests
import json
import spacy
import trafilatura
from neo4j import GraphDatabase
from google.colab import userdata
from datetime import datetime, timezone
from groq import Groq

print("✅ All imports successful!")

# Load secrets
NEWSAPI_KEY = userdata.get('NEWSAPI_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')

print("✅ Secrets loaded!")

# Test Neo4j connection
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print("✅ Neo4j connected!")

# Test Groq connection
client = Groq(api_key=GROQ_API_KEY)
test = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say: Prajna is online!"}],
    max_tokens=20
)
print(f"✅ Groq connected! Response: {test.choices[0].message.content}")

# Test NewsAPI
resp = requests.get(f"https://newsapi.org/v2/everything?q=India&pageSize=1&apiKey={NEWSAPI_KEY}")
print(f"✅ NewsAPI connected! Status: {resp.json()['status']}")

print("\n🚀 ALL SYSTEMS GO — PRAJNA IS READY TO BUILD!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 16.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 55.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [23]:
# ============================================
# CELL 2 — Layer 1: Data Ingestion + Neo4j
# FIXED: Fast trafilatura with short timeout
# ============================================

import requests
import json
import spacy
import trafilatura
from trafilatura.settings import use_config
from neo4j import GraphDatabase
from google.colab import userdata
from datetime import datetime, timezone
import concurrent.futures

nlp = spacy.load("en_core_web_sm")

# ── Credentials ──
NEWSAPI_KEY  = userdata.get("NEWSAPI_KEY")
NEO4J_URI    = userdata.get("NEO4J_URI")
NEO4J_USER   = userdata.get("NEO4J_USERNAME")
NEO4J_PASS   = userdata.get("NEO4J_PASSWORD")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

# ── Trafilatura config — FAST timeouts ──
traf_config = use_config()
traf_config.set("DEFAULT", "DOWNLOAD_TIMEOUT", "5")   # 5s max per URL (was 30s)
traf_config.set("DEFAULT", "MAX_REDIRECTS", "2")

# ── Topics ──
DOMAIN_TOPICS = {
    "GEOPOLITICS": [
        "India geopolitics 2026", "India China relations",
        "India Pakistan border", "India Iran oil energy",
        "India Russia defense", "India Middle East policy",
        "Taliban Afghanistan conflict", "Israel Iran war",
        "India USA relations 2026", "India Bangladesh relations",
        "India Nepal border", "BRICS India 2026", "Quad alliance Indo-Pacific",
    ],
    "DOMESTIC": [
        "India elections 2026", "India parliament budget 2026",
        "India BJP Congress politics", "India supreme court judgment",
    ],
    "ECONOMICS": [
        "India economy GDP 2026", "India stock market Sensex",
        "India trade exports", "India inflation RBI policy",
        "India FDI foreign investment 2026",
    ],
    "TECHNOLOGY": [
        "India semiconductor chip manufacturing", "India AI policy",
        "India space ISRO 2026", "India defence procurement",
        "India cybersecurity threat",
    ],
    "SOCIETY": [
        "India infrastructure smart city", "India climate environment",
        "India health policy", "India education skill development",
        "India startup ecosystem 2026",
    ],
}

# Flatten for ingestion
ALL_TOPICS = [(topic, domain)
              for domain, topics in DOMAIN_TOPICS.items()
              for topic in topics]

def get_week_key():
    now = datetime.now(timezone.utc)
    return f"{now.year}-W{now.strftime('%W')}"

# ── Entity normalization map ──
ENTITY_ALIASES = {
    "Indian": "India", "Indians": "India",
    "US": "United States", "U.S.": "United States",
    "U.S.A.": "United States", "USA": "United States",
    "America": "United States", "American": "United States",
    "Pakistani": "Pakistan", "Pakistanis": "Pakistan",
    "Iranian": "Iran", "Iranians": "Iran",
    "Israeli": "Israel", "Israelis": "Israel",
    "Chinese": "China", "Russians": "Russia",
    "Russian": "Russia", "Afghan": "Afghanistan",
    "Afghans": "Afghanistan", "the United States": "United States",
    "the Middle East": "Middle East",
    "the European Union": "European Union",
}

def normalize_entity(text):
    return ENTITY_ALIASES.get(text.strip(), text.strip())

# ── Step 1: Fetch articles ──
def fetch_articles(topics_with_domains, api_key, page_size=10):
    all_articles = []
    seen_urls = set()
    for topic, domain in topics_with_domains:
        r = requests.get("https://newsapi.org/v2/everything", params={
            "q": topic, "language": "en",
            "sortBy": "publishedAt",
            "pageSize": page_size, "apiKey": api_key
        })
        if r.status_code == 200:
            results = r.json().get("articles", [])
            new = 0
            for a in results:
                url = a.get("url","")
                if url not in seen_urls:
                    seen_urls.add(url)
                    a["domain"] = domain  # tag with domain
                    all_articles.append(a)
                    new += 1
            print(f"  ✅ {topic}: {new} new articles")
        else:
            print(f"  ❌ {topic}: failed ({r.status_code})")
    return all_articles

# ── Step 2: Scrape full text (FAST — 5s timeout, parallel) ──
def scrape_full_text(url):
    """Scrape with hard 5-second timeout. Returns None on any failure."""
    try:
        downloaded = trafilatura.fetch_url(url, config=traf_config)
        if downloaded:
            text = trafilatura.extract(downloaded, config=traf_config)
            return text
    except:
        pass
    return None

def extract_entities(articles):
    """Extract entities — uses full text if scraped, else title+description."""
    VALID_LABELS = {"GPE","ORG","PERSON","NORP","LOC","EVENT"}
    entities_per_article = []
    scraped = 0

    # Parallel scraping — 10 workers, each with 5s timeout
    urls = [a.get("url","") for a in articles]
    full_texts = {}

    print(f"  🌐 Scraping full text (fast mode — 5s timeout per URL)...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        future_to_url = {executor.submit(scrape_full_text, url): url
                         for url in urls if url}
        for i, future in enumerate(concurrent.futures.as_completed(future_to_url)):
            url = future_to_url[future]
            try:
                text = future.result(timeout=8)  # hard kill at 8s
                if text:
                    full_texts[url] = text
                    scraped += 1
            except:
                pass
            if (i+1) % 50 == 0:
                print(f"  📄 Scraped {i+1}/{len(urls)} ({scraped} successful)")

    print(f"  ✅ Full text: {scraped}/{len(urls)} articles scraped")

    # Extract entities
    for article in articles:
        url  = article.get("url","")
        text = full_texts.get(url) or \
               f"{article.get('title','')} {article.get('description','')}"

        doc = nlp(text[:5000])  # cap at 5000 chars to keep NLP fast
        entities = []
        seen = set()
        for ent in doc.ents:
            if ent.label_ not in VALID_LABELS: continue
            normalized = normalize_entity(ent.text.strip())
            if len(normalized) < 2: continue
            if len(normalized) > 40: continue  # skip headline fragments
            if normalized in seen: continue
            seen.add(normalized)
            entities.append((normalized, ent.label_))
        entities_per_article.append(entities)

    return entities_per_article

# ── Step 3: Populate graph ──
def populate_graph(articles, entities_per_article):
    week_key = get_week_key()
    print(f"\n📊 Populating Neo4j graph...")
    print(f"📅 Writing to graph for week: {week_key}")

    with driver.session() as session:
        # Articles
        for article in articles:
            session.run("""
                MERGE (a:Article {url: $url})
                SET a.title=$title, a.source=$source,
                    a.published=$published, a.week=$week,
                    a.domain=$domain
            """, {
                "url":       article.get("url","unknown"),
                "title":     article.get("title",""),
                "source":    article.get("source",{}).get("name","Unknown"),
                "published": article.get("publishedAt",""),
                "week":      week_key,
                "domain":    article.get("domain","GENERAL")
            })
        print(f"  ✅ {len(articles)} articles written")

        # Entities + MENTIONS
        for article, entities in zip(articles, entities_per_article):
            for entity_text, entity_label in entities:
                session.run("""
                    MERGE (e:Entity {name: $name})
                    SET e.type = $type
                    WITH e
                    MATCH (a:Article {url: $url})
                    MERGE (a)-[:MENTIONS]->(e)
                """, {
                    "name":  entity_text,
                    "type":  entity_label,
                    "url":   article.get("url","unknown")
                })
        print(f"  ✅ Entities + MENTIONS written")

        # CO_OCCURS_WITH edges with weekly tracking
        pair_count = 0
        for article, entities in zip(articles, entities_per_article):
            unique_entities = list(set([e[0] for e in entities]))
            domain = article.get("domain","GENERAL")

            for i in range(len(unique_entities)):
                for j in range(i+1, len(unique_entities)):
                    e1, e2 = unique_entities[i], unique_entities[j]

                    result = session.run("""
                        MATCH (a:Entity {name:$e1})-[r:CO_OCCURS_WITH]-(b:Entity {name:$e2})
                        RETURN r.weekly_counts_json AS wcj, r.count AS cnt,
                               r.domains_json AS dj
                    """, {"e1": e1, "e2": e2}).single()

                    if result:
                        wc  = json.loads(result["wcj"]) if result["wcj"] else {}
                        cnt = result["cnt"] or 0
                        dj  = json.loads(result["dj"]) if result["dj"] else {}
                        wc[week_key]  = wc.get(week_key, 0) + 1
                        dj[domain]    = dj.get(domain, 0) + 1
                        session.run("""
                            MATCH (a:Entity {name:$e1})-[r:CO_OCCURS_WITH]-(b:Entity {name:$e2})
                            SET r.count=$cnt,
                                r.weekly_counts_json=$wcj,
                                r.domains_json=$dj
                        """, {"e1":e1,"e2":e2,
                              "cnt":cnt+1,
                              "wcj":json.dumps(wc),
                              "dj":json.dumps(dj)})
                    else:
                        session.run("""
                            MATCH (a:Entity {name:$e1})
                            MATCH (b:Entity {name:$e2})
                            MERGE (a)-[r:CO_OCCURS_WITH]-(b)
                            SET r.count=1,
                                r.first_seen=$week,
                                r.weekly_counts_json=$wcj,
                                r.domains_json=$dj
                        """, {"e1":e1,"e2":e2,"week":week_key,
                              "wcj":json.dumps({week_key:1}),
                              "dj":json.dumps({domain:1})})
                    pair_count += 1

        print(f"  ✅ {pair_count} entity pairs processed")

    print(f"\n🎉 Graph populated for week {week_key}!")

# ══════════════════════════════════════════
# RUN
# ══════════════════════════════════════════

# Clear graph first
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
print("🗑️  Graph cleared\n")

print(f"✅ {len(ALL_TOPICS)} topics loaded across 5 domains")
print("📡 Fetching articles from NewsAPI...")
articles = fetch_articles(ALL_TOPICS, NEWSAPI_KEY, page_size=10)
print(f"\n✅ Total unique articles: {len(articles)}")

print("\n🔍 Extracting entities (parallel full-text scraping)...")
entities_per_article = extract_entities(articles)

populate_graph(articles, entities_per_article)

🗑️  Graph cleared

✅ 32 topics loaded across 5 domains
📡 Fetching articles from NewsAPI...
  ❌ India geopolitics 2026: failed (429)
  ❌ India China relations: failed (429)
  ❌ India Pakistan border: failed (429)
  ❌ India Iran oil energy: failed (429)
  ❌ India Russia defense: failed (429)
  ❌ India Middle East policy: failed (429)
  ❌ Taliban Afghanistan conflict: failed (429)
  ❌ Israel Iran war: failed (429)
  ❌ India USA relations 2026: failed (429)
  ❌ India Bangladesh relations: failed (429)
  ❌ India Nepal border: failed (429)
  ❌ BRICS India 2026: failed (429)
  ❌ Quad alliance Indo-Pacific: failed (429)
  ❌ India elections 2026: failed (429)
  ❌ India parliament budget 2026: failed (429)
  ❌ India BJP Congress politics: failed (429)
  ❌ India supreme court judgment: failed (429)
  ❌ India economy GDP 2026: failed (429)
  ❌ India stock market Sensex: failed (429)
  ❌ India trade exports: failed (429)
  ❌ India inflation RBI policy: failed (429)
  ❌ India FDI foreign investment 

In [3]:
# ── RECONNECT Neo4j (fixes NotALeader) ──
driver.close()
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))
print("✅ Neo4j reconnected")

# ── POPULATE GRAPH (batched — fast) ──
from collections import defaultdict

def populate_graph_fast(articles, entities_per_article):
    week_key = get_week_key()
    print(f"📊 Populating Neo4j — week {week_key}")

    with driver.session() as session:

        # Articles batch
        session.run("""
            UNWIND $batch AS row
            MERGE (a:Article {url: row.url})
            SET a.title=row.title, a.source=row.source,
                a.published=row.published, a.week=row.week,
                a.domain=row.domain
        """, {"batch": [{
            "url":       a.get("url","unknown"),
            "title":     a.get("title",""),
            "source":    a.get("source",{}).get("name","Unknown"),
            "published": a.get("publishedAt",""),
            "week":      week_key,
            "domain":    a.get("domain","GENERAL")
        } for a in articles]})
        print(f"  ✅ {len(articles)} articles written")

        # Entities + MENTIONS batch
        mentions = []
        for article, entities in zip(articles, entities_per_article):
            for entity_text, entity_label in entities:
                mentions.append({
                    "name": entity_text,
                    "type": entity_label,
                    "url":  article.get("url","unknown")
                })

        # Write mentions in chunks of 500
        for i in range(0, len(mentions), 500):
            session.run("""
                UNWIND $batch AS row
                MERGE (e:Entity {name: row.name})
                SET e.type = row.type
                WITH e, row
                MATCH (a:Article {url: row.url})
                MERGE (a)-[:MENTIONS]->(e)
            """, {"batch": mentions[i:i+500]})
        print(f"  ✅ {len(mentions)} entity mentions written")

        # Collect all pairs in Python first
        pair_counts  = defaultdict(int)
        pair_domains = defaultdict(lambda: defaultdict(int))

        for article, entities in zip(articles, entities_per_article):
            unique = list(set([e[0] for e in entities]))
            domain = article.get("domain","GENERAL")
            for i in range(len(unique)):
                for j in range(i+1, len(unique)):
                    key = tuple(sorted([unique[i], unique[j]]))
                    pair_counts[key]  += 1
                    pair_domains[key][domain] += 1

        print(f"  📊 {len(pair_counts)} unique entity pairs")

        # Write pairs in batches of 200
        pairs_list = list(pair_counts.items())
        for batch_start in range(0, len(pairs_list), 200):
            batch = pairs_list[batch_start:batch_start+200]
            session.run("""
                UNWIND $batch AS row
                MATCH (a:Entity {name: row.e1})
                MATCH (b:Entity {name: row.e2})
                MERGE (a)-[r:CO_OCCURS_WITH]-(b)
                ON CREATE SET
                    r.count              = row.cnt,
                    r.first_seen         = row.week,
                    r.weekly_counts_json = row.wcj,
                    r.domains_json       = row.dj
                ON MATCH SET
                    r.count              = r.count + row.cnt
            """, {"batch": [{
                "e1":   pair[0],
                "e2":   pair[1],
                "cnt":  count,
                "week": week_key,
                "wcj":  json.dumps({week_key: count}),
                "dj":   json.dumps(dict(pair_domains[pair]))
            } for pair, count in batch]})
            print(f"  ✅ Pairs: {min(batch_start+200, len(pairs_list))}/{len(pairs_list)}")

    print(f"\n🎉 Done! Graph populated for week {week_key}")

populate_graph_fast(articles, entities_per_article)

✅ Neo4j reconnected
📊 Populating Neo4j — week 2026-W09
  ✅ 260 articles written
  ✅ 6184 entity mentions written
  📊 77297 unique entity pairs
  ✅ Pairs: 200/77297
  ✅ Pairs: 400/77297
  ✅ Pairs: 600/77297
  ✅ Pairs: 800/77297
  ✅ Pairs: 1000/77297
  ✅ Pairs: 1200/77297
  ✅ Pairs: 1400/77297
  ✅ Pairs: 1600/77297
  ✅ Pairs: 1800/77297
  ✅ Pairs: 2000/77297
  ✅ Pairs: 2200/77297
  ✅ Pairs: 2400/77297
  ✅ Pairs: 2600/77297
  ✅ Pairs: 2800/77297
  ✅ Pairs: 3000/77297
  ✅ Pairs: 3200/77297
  ✅ Pairs: 3400/77297
  ✅ Pairs: 3600/77297
  ✅ Pairs: 3800/77297
  ✅ Pairs: 4000/77297
  ✅ Pairs: 4200/77297
  ✅ Pairs: 4400/77297
  ✅ Pairs: 4600/77297
  ✅ Pairs: 4800/77297
  ✅ Pairs: 5000/77297
  ✅ Pairs: 5200/77297
  ✅ Pairs: 5400/77297
  ✅ Pairs: 5600/77297
  ✅ Pairs: 5800/77297
  ✅ Pairs: 6000/77297
  ✅ Pairs: 6200/77297
  ✅ Pairs: 6400/77297
  ✅ Pairs: 6600/77297
  ✅ Pairs: 6800/77297
  ✅ Pairs: 7000/77297
  ✅ Pairs: 7200/77297
  ✅ Pairs: 7400/77297
  ✅ Pairs: 7600/77297
  ✅ Pairs: 7800/77297
  ✅

In [16]:
# ============================================
# CELL 4 — Graph Cleanup + Backfill
# Run once after Cell 3, then as needed
# ============================================

# ── Cleanup known bad entities ──
with driver.session() as session:
    session.run("MATCH (e:Entity {name:'China'}) SET e.type='GPE'")
    session.run("MATCH (e:Entity {name:'AI', type:'GPE'}) DETACH DELETE e")
    session.run("""
        MATCH (keep:Entity {name:'Trump'})
        MATCH (dup:Entity {name:'Donald Trump'})
        WITH keep, dup
        MATCH (dup)-[r:CO_OCCURS_WITH]-(other:Entity)
        WHERE other <> keep
        MERGE (keep)-[r2:CO_OCCURS_WITH]-(other)
        ON CREATE SET r2.count=r.count,
                      r2.weekly_counts_json=r.weekly_counts_json
        ON MATCH SET  r2.count=r2.count+r.count
        WITH dup DETACH DELETE dup
    """)
    session.run("""
        MATCH (keep:Entity {name:'United States'})
        MATCH (dup:Entity {name:'Washington'})
        WITH keep, dup
        MATCH (dup)-[r:CO_OCCURS_WITH]-(other:Entity)
        WHERE other <> keep
        MERGE (keep)-[r2:CO_OCCURS_WITH]-(other)
        ON CREATE SET r2.count=r.count
        ON MATCH SET  r2.count=r2.count+r.count
        WITH dup DETACH DELETE dup
    """)
    session.run("""
        MATCH (e:Entity {name:'the Strait of Hormuz'})
        SET e.name='Strait of Hormuz'
    """)
print("✅ Graph cleanup done")

# ── Quick graph stats ──
with driver.session() as session:
    articles = session.run("MATCH (a:Article) RETURN count(a) as c").single()["c"]
    entities = session.run("MATCH (e:Entity) RETURN count(e) as c").single()["c"]
    rels     = session.run("MATCH ()-[r:CO_OCCURS_WITH]->() RETURN count(r) as c").single()["c"]
    print(f"\n📊 Graph Status:")
    print(f"   Articles:      {articles}")
    print(f"   Entities:      {entities}")
    print(f"   Relationships: {rels}")

    print(f"\n🌍 Top 15 Entities:")
    result = session.run("""
        MATCH (e:Entity)
        WITH e, COUNT{(e)--()} as conn
        ORDER BY conn DESC LIMIT 15
        RETURN e.name as name, e.type as type, conn
    """)
    for r in result:
        print(f"   {r['name']:<25} {r['type']:<10} {r['conn']}")

    print(f"\n📅 Weeks in graph:")
    result = session.run("""
        MATCH (a:Article)
        RETURN a.week as week, count(a) as articles
        ORDER BY week
    """)
    for r in result:
        print(f"   {r['week']}: {r['articles']} articles")


✅ Graph cleanup done

📊 Graph Status:
   Articles:      321
   Entities:      3485
   Relationships: 78188

🌍 Top 15 Entities:
   India                     NORP       2774
   United States             GPE        2269
   Iran                      GPE        1354
   Israel                    GPE        1235
   China                     GPE        1122
   Middle East               LOC        1073
   Trump                     ORG        940
   Russia                    GPE        746
   Japan                     GPE        741
   Europe                    LOC        733
   Pakistan                  GPE        620
   Australia                 GPE        572
   Saudi Arabia              GPE        551
   Qatar                     GPE        545
   Strait of Hormuz          LOC        526

📅 Weeks in graph:
   2026-W05: 53 articles
   2026-W06: 25 articles
   2026-W09: 243 articles


In [17]:
with driver.session() as session:
    session.run("MATCH (e:Entity {name:'India'}) SET e.type='GPE'")
    session.run("MATCH (e:Entity {name:'Trump'}) SET e.type='PERSON'")
print("✅ Fixed")

✅ Fixed


In [ ]:
# ============================================
# CELL 3 — Explore what's in the graph (fixed)
# ============================================

with driver.session() as session:

    # Top entities by connection count
    print("🌍 TOP ENTITIES IN PRAJNA'S GRAPH:")
    print("=" * 50)
    result = session.run("""
        MATCH (e:Entity)
        WITH e, COUNT { (e)--() } as connections
        ORDER BY connections DESC
        LIMIT 20
        RETURN e.name as entity, e.type as type, connections
    """)
    for r in result:
        print(f"   {r['entity']:<30} [{r['type']}]  — {r['connections']} connections")

    # Top countries
    print("\n🗺️ COUNTRIES/PLACES IN THE GRAPH:")
    print("=" * 50)
    result2 = session.run("""
        MATCH (e:Entity)
        WHERE e.type IN ['GPE', 'LOC']
        WITH e, COUNT { (e)--() } as connections
        ORDER BY connections DESC
        LIMIT 15
        RETURN e.name as place, connections
    """)
    for r in result2:
        print(f"   {r['place']:<30} — {r['connections']} connections")

    # Top organizations
    print("\n🏢 ORGANIZATIONS IN THE GRAPH:")
    print("=" * 50)
    result3 = session.run("""
        MATCH (e:Entity)
        WHERE e.type = 'ORG'
        WITH e, COUNT { (e)--() } as connections
        ORDER BY connections DESC
        LIMIT 15
        RETURN e.name as org, connections
    """)
    for r in result3:
        print(f"   {r['org']:<30} — {r['connections']} connections")

    # Sample relationships — who is India connected to?
    print("\n🇮🇳 INDIA IS CONNECTED TO:")
    print("=" * 50)
    result4 = session.run("""
        MATCH (india:Entity)-[r:CO_OCCURS_WITH]-(other:Entity)
        WHERE india.name IN ['India', 'Indian']
        ORDER BY r.count DESC
        LIMIT 15
        RETURN other.name as entity, other.type as type, r.count as strength
    """)
    for r in result4:
        print(f"   {r['entity']:<30} [{r['type']}]  — strength: {r['strength']}")

🌍 TOP ENTITIES IN PRAJNA'S GRAPH:
   Iran                           [GPE]  — 122 connections
   US                             [GPE]  — 80 connections
   Israel                         [GPE]  — 70 connections
   India                          [GPE]  — 62 connections
   Afghanistan                    [GPE]  — 47 connections
   Pakistan                       [GPE]  — 46 connections
   U.S.                           [GPE]  — 36 connections
   Middle East                    [LOC]  — 29 connections
   Russia                         [GPE]  — 28 connections
   Israeli                        [NORP]  — 26 connections
   China                          [GPE]  — 25 connections
   the Middle East                [LOC]  — 22 connections
   Kabul                          [GPE]  — 21 connections
   Pakistani                      [NORP]  — 21 connections
   Iranian                        [NORP]  — 20 connections
   the United States              [GPE]  — 19 connections
   Taliban                        

In [ ]:
# ============================================
# CELL 4 — Layer 4: AI Reasoning with Groq
# Ask Prajna anything — get cited intel briefs
# ============================================

from groq import Groq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

def query_graph_context(question, limit=30):
    """Pull relevant context from Neo4j based on question keywords"""
    # Extract keywords from question (simple approach)
    keywords = [w for w in question.lower().split()
                if len(w) > 3 and w not in
                ['what', 'how', 'does', 'india', 'with', 'that', 'this', 'will', 'from', 'about']]

    context_data = []

    with driver.session() as session:
        # Get entities related to keywords
        for keyword in keywords[:3]:  # top 3 keywords
            result = session.run("""
                MATCH (e:Entity)
                WHERE toLower(e.name) CONTAINS $keyword
                WITH e
                MATCH (e)-[r:CO_OCCURS_WITH]-(other:Entity)
                RETURN e.name as entity1, other.name as entity2,
                       r.count as strength, e.type as type1, other.type as type2
                ORDER BY r.count DESC
                LIMIT $limit
            """, keyword=keyword, limit=limit)

            for r in result:
                context_data.append(
                    f"{r['entity1']} ({r['type1']}) ↔ {r['entity2']} ({r['type2']}) "
                    f"[co-occurrence strength: {r['strength']}]"
                )

        # Also get relevant articles
        result2 = session.run("""
            MATCH (a:Article)-[:MENTIONS]->(e:Entity)
            WHERE any(keyword IN $keywords WHERE toLower(e.name) CONTAINS keyword)
            RETURN DISTINCT a.title as title, a.source as source, a.published as published
            ORDER BY a.published DESC
            LIMIT 10
        """, keywords=keywords)

        articles_context = []
        for r in result2:
            articles_context.append(
                f"[{r['source']}] {r['title']} ({r['published'][:10] if r['published'] else 'recent'})"
            )

    return context_data, articles_context


def ask_prajna(question):
    """Ask Prajna a strategic intelligence question"""
    print(f"\n🧠 PRAJNA INTELLIGENCE BRIEF")
    print(f"{'='*60}")
    print(f"❓ Query: {question}")
    print(f"{'='*60}")

    # Get context from graph
    graph_context, article_context = query_graph_context(question)

    if not graph_context and not article_context:
        print("⚠️  Limited data for this query. Try running ingestion again.")
        return

    # Build context string
    context_str = ""
    if graph_context:
        context_str += "KNOWLEDGE GRAPH CONNECTIONS:\n"
        context_str += "\n".join(graph_context[:20])

    if article_context:
        context_str += "\n\nRELEVANT NEWS ARTICLES:\n"
        context_str += "\n".join(article_context)

    # Ask Groq with context
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are Prajna — India's AI-powered strategic intelligence analyst.

Your job is to provide concise, actionable intelligence briefs for Indian policy makers and government officials.

Rules:
1. ONLY use the provided graph context and articles to answer
2. Cite your sources (mention article sources and entity connections)
3. Structure every response as:
   📌 SITUATION SUMMARY (2-3 lines)
   🔗 KEY CONNECTIONS FROM GRAPH (bullet points)
   ⚡ STRATEGIC IMPLICATIONS FOR INDIA (2-3 bullet points)
   📰 BASED ON: (list sources)
4. Be specific, factual, and policy-relevant
5. If context is insufficient, say so clearly"""
            },
            {
                "role": "user",
                "content": f"""Based on this intelligence from Prajna's knowledge graph:

{context_str}

Answer this strategic question: {question}"""
            }
        ],
        max_tokens=600
    )

    print(response.choices[0].message.content)
    print(f"\n{'='*60}")
    print(f"💡 Graph connections analysed: {len(graph_context)}")
    print(f"📰 Articles referenced: {len(article_context)}")


# ── TEST WITH 3 DEMO QUERIES ─────────────────────────────────────

ask_prajna("What is India's relationship with Iran and what are the strategic risks?")

ask_prajna("How does the situation in Pakistan affect India's security?")

ask_prajna("What is India's position in the semiconductor and technology space?")


🧠 PRAJNA INTELLIGENCE BRIEF
❓ Query: What is India's relationship with Iran and what are the strategic risks?
📌 SITUATION SUMMARY:
Tensions are escalating in the Middle East following the death of Ayatollah Ali Khamenei, Iran's supreme leader. The US and Israel have reportedly conducted strikes on Iran, which could have implications for global oil trade and stability in the region.

🔗 KEY CONNECTIONS FROM GRAPH:
- Iran is connected to Israel with a moderate strength of 16.
- Iran is connected to the US with a moderate strength of 14.
- Iran is connected to India with a weak strength of 5, indicating a limited relationship.
- The Strait of Hormuz, a critical global oil trade route, is connected to Iran.
- The Middle East, a region of significant importance to India's foreign policy, is connected to Iran.

⚡ STRATEGIC IMPLICATIONS FOR INDIA:
- India may face energy security risks due to potential disruptions in global oil trade through the Strait of Hormuz.
- India-IR Iran relationship 

In [ ]:
# ============================================
# CELL 5 — Visual Graph (PyVis)
# Creates interactive graph you can screenshot
# ============================================

from pyvis.network import Network
import json

# Color scheme by entity type
TYPE_COLORS = {
    "GPE": "#7c3aed",      # Purple — countries
    "ORG": "#06b6d4",      # Teal — organizations
    "PERSON": "#f97316",   # Orange — people
    "NORP": "#fbbf24",     # Gold — nationalities
    "LOC": "#4ade80",      # Green — locations
    "EVENT": "#f43f5e",    # Red — events
    "MONEY": "#a3e635",    # Lime — money
}

TYPE_LABELS = {
    "GPE": "🌍 Country",
    "ORG": "🏢 Org",
    "PERSON": "👤 Person",
    "NORP": "👥 Group",
    "LOC": "📍 Location",
    "EVENT": "📅 Event",
    "MONEY": "💰 Money",
}

print("🔄 Building visualization...")

# Fetch top entities and their connections
with driver.session() as session:
    # Get top 40 most connected entities
    entities_result = session.run("""
        MATCH (e:Entity)
        WITH e, COUNT { (e)--() } as connections
        ORDER BY connections DESC
        LIMIT 40
        RETURN e.name as name, e.type as type, connections
    """)
    top_entities = {r['name']: r for r in entities_result}

    # Get relationships between top entities only
    rels_result = session.run("""
        MATCH (e1:Entity)-[r:CO_OCCURS_WITH]-(e2:Entity)
        WHERE e1.name IN $names AND e2.name IN $names
        AND r.count >= 2
        RETURN e1.name as source, e2.name as target, r.count as weight
    """, names=list(top_entities.keys()))
    relationships = list(rels_result)

print(f"📊 Visualizing {len(top_entities)} entities and {len(relationships)} connections...")

# Build PyVis network
net = Network(
    height="700px",
    width="100%",
    bgcolor="#06040f",
    font_color="#f0eeff",
    notebook=True,
    cdn_resources='in_line'
)

# Configure physics for nice layout
net.set_options("""
{
  "physics": {
    "forceAtlas2Based": {
      "gravitationalConstant": -80,
      "centralGravity": 0.01,
      "springLength": 120,
      "springConstant": 0.08
    },
    "solver": "forceAtlas2Based",
    "stabilization": { "iterations": 150 }
  },
  "nodes": {
    "borderWidth": 2,
    "shadow": true
  },
  "edges": {
    "smooth": { "type": "continuous" },
    "shadow": true
  }
}
""")

# Add nodes
for name, data in top_entities.items():
    entity_type = data['type']
    connections = data['connections']
    color = TYPE_COLORS.get(entity_type, "#8b7aa8")
    label = TYPE_LABELS.get(entity_type, "◆")

    # Size nodes by connection count
    size = min(15 + connections * 1.5, 50)

    # Highlight India specially
    if name in ['India', 'Indian']:
        color = "#ff6b35"
        size = 55

    net.add_node(
        name,
        label=name,
        title=f"{label} | {connections} connections",
        color=color,
        size=size,
        font={"size": 12, "color": "#ffffff"}
    )

# Add edges
for r in relationships:
    if r['source'] in top_entities and r['target'] in top_entities:
        weight = r['weight']
        net.add_edge(
            r['source'],
            r['target'],
            value=weight,
            title=f"Co-occurrence: {weight}",
            color={"color": "rgba(124,58,237,0.4)"}
        )

# Save and display
net.show("prajna_graph.html")
print("✅ Graph saved as prajna_graph.html!")
print("\n📸 SCREENSHOT INSTRUCTIONS:")
print("   The interactive graph is displayed below.")
print("   - Drag nodes to rearrange")
print("   - Scroll to zoom in/out")
print("   - Hover over nodes to see details")
print("   - Screenshot this for your PPT!")
print("\n🎨 LEGEND:")
for etype, color in TYPE_COLORS.items():
    print(f"   {TYPE_LABELS.get(etype, etype)}: {color}")

🔄 Building visualization...
📊 Visualizing 40 entities and 178 connections...
prajna_graph.html
✅ Graph saved as prajna_graph.html!

📸 SCREENSHOT INSTRUCTIONS:
   The interactive graph is displayed below.
   - Drag nodes to rearrange
   - Scroll to zoom in/out
   - Hover over nodes to see details
   - Screenshot this for your PPT!

🎨 LEGEND:
   🌍 Country: #7c3aed
   🏢 Org: #06b6d4
   👤 Person: #f97316
   👥 Group: #fbbf24
   📍 Location: #4ade80
   📅 Event: #f43f5e
   💰 Money: #a3e635


In [ ]:
# ============================================
# CELL 6 — Download Graph
# ============================================

from google.colab import files
files.download('prajna_graph.html')

In [5]:
# ============================================
# CELL 7 — Install Streamlit + ngrok for Colab
# ============================================
!pip install streamlit pyngrok -q

# Set up ngrok (gives your app a public URL)
from pyngrok import ngrok

# You need a free ngrok account — sign up at ngrok.com
# Then get your authtoken from: dashboard.ngrok.com/get-started/your-authtoken
# Paste it below:
NGROK_TOKEN = "3ALaOKHXsuRme9qCxyIivo6aidG_75Yki5wLqrfRJhB83kqSV"
ngrok.set_auth_token(NGROK_TOKEN)

print("✅ Streamlit + ngrok ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 88.7 MB/s eta 0:00:00
✅ Streamlit + ngrok ready!


In [6]:
# ============================================
# CELL 8 — Write the Streamlit app to a file
# ============================================

app_code = '''
import streamlit as st
from neo4j import GraphDatabase
from groq import Groq
from pyvis.network import Network
import streamlit.components.v1 as components
import os, json
from collections import defaultdict
from datetime import datetime, timezone

# ── Connections ──
NEO4J_URI      = os.environ.get("NEO4J_URI")
NEO4J_USERNAME = os.environ.get("NEO4J_USERNAME")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
GROQ_API_KEY   = os.environ.get("GROQ_API_KEY")

@st.cache_resource
def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

@st.cache_resource
def get_groq_client():
    return Groq(api_key=GROQ_API_KEY)

driver      = get_driver()
groq_client = get_groq_client()

def get_week_key():
    now = datetime.now(timezone.utc)
    return f"{now.year}-W{now.strftime('%W')}"

def ask_groq(prompt):
    r = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}],
        max_tokens=600, temperature=0.3
    )
    return r.choices[0].message.content

@st.cache_data(ttl=300)
def get_stats():
    with driver.session() as session:
        articles = session.run("MATCH (a:Article) RETURN count(a) as c").single()["c"]
        entities = session.run("MATCH (e:Entity) RETURN count(e) as c").single()["c"]
        rels     = session.run("MATCH ()-[r:CO_OCCURS_WITH]->() RETURN count(r) as c").single()["c"]
    return articles, entities, rels

def get_graph_context(keywords):
    with driver.session() as session:
        result = session.run("""
            MATCH (e:Entity)-[r:CO_OCCURS_WITH]-(other:Entity)
            WHERE any(kw IN $keywords WHERE toLower(e.name) CONTAINS toLower(kw))
            RETURN e.name AS e1, other.name AS e2, r.count AS strength
            ORDER BY r.count DESC LIMIT 20
        """, {"keywords": keywords})
        return [(r["e1"], r["e2"], r["strength"]) for r in result]

def get_articles(keywords):
    with driver.session() as session:
        result = session.run("""
            MATCH (a:Article)-[:MENTIONS]->(e:Entity)
            WHERE any(kw IN $keywords WHERE toLower(e.name) CONTAINS toLower(kw))
            RETURN DISTINCT a.title AS title, a.source AS source
            LIMIT 5
        """, {"keywords": keywords})
        return [(r["title"], r["source"]) for r in result]

def get_trajectory(e1, e2):
    with driver.session() as session:
        result = session.run("""
            MATCH (a:Entity {name:$e1})-[r:CO_OCCURS_WITH]-(b:Entity {name:$e2})
            RETURN r.weekly_counts_json AS wcj, r.count AS total
        """, {"e1": e1, "e2": e2}).single()
    if not result or not result["wcj"]:
        return None, None
    wc = json.loads(result["wcj"])
    return sorted(wc.items()), result["total"]

def find_path(e1, e2):
    with driver.session() as session:
        result = session.run("""
            MATCH path = shortestPath(
                (a:Entity {name:$e1})-[:CO_OCCURS_WITH*1..4]-(b:Entity {name:$e2})
            )
            RETURN [node IN nodes(path) | node.name] AS nodes,
                   [rel IN relationships(path) | rel.count] AS strengths,
                   length(path) AS hops
        """, {"e1": e1, "e2": e2}).single()
    if not result:
        return None
    return {"nodes": result["nodes"], "strengths": result["strengths"], "hops": result["hops"]}

BLOCKLIST = {
    "Supreme","Asian","Islam","American","European","Western","Eastern",
    "T20 World Cup","West Indies","BBC","CNN","Reuters","Bloomberg",
    "NDTV","Mint","Hindu","Express","ANI","PTI","AFP","AP","Wire","Tribune","Times"
}

def detect_surges(threshold=1.5, top_n=5):
    week_key = get_week_key()
    with driver.session() as session:
        results = session.run("""
            MATCH (e:Entity)-[r:CO_OCCURS_WITH]-()
            WHERE r.weekly_counts_json IS NOT NULL
            RETURN e.name AS entity, e.type AS type,
                   r.weekly_counts_json AS wcj, r.count AS total
        """).data()

    entity_weekly = defaultdict(lambda: defaultdict(int))
    entity_total  = defaultdict(int)
    entity_type   = {}

    for row in results:
        e = row["entity"]
        entity_type[e] = row["type"]
        entity_total[e] += row["total"] or 0
        wc = json.loads(row["wcj"]) if row["wcj"] else {}
        for week, count in wc.items():
            entity_weekly[e][week] += count

    surges = []
    for entity, weekly in entity_weekly.items():
        if entity in BLOCKLIST: continue
        if entity_total[entity] < 8: continue
        if len(entity) > 35: continue
        if any(c.isdigit() for c in entity): continue
        if entity_type.get(entity) not in {"GPE","ORG","PERSON","NORP"}: continue

        weeks     = sorted(weekly.keys())
        latest_wk = weeks[-1]
        if latest_wk != week_key: continue

        latest   = weekly[latest_wk]
        hist     = [weekly[w] for w in weeks if w != latest_wk]
        baseline = sum(hist)/len(hist) if hist else entity_total[entity]/2
        if baseline == 0: continue

        ratio = latest / baseline
        if ratio >= threshold:
            surges.append({
                "entity":   entity,
                "type":     entity_type.get(entity,""),
                "latest":   latest,
                "baseline": round(baseline,1),
                "ratio":    round(ratio,2),
                "total":    entity_total[entity]
            })

    surges.sort(key=lambda x: x["ratio"], reverse=True)
    return surges[:top_n]

def build_graph_visual(keyword=None):
    TYPE_COLORS = {
        "GPE":"#C8A96E","ORG":"#6B8CAE","PERSON":"#A8C5A0",
        "NORP":"#B8956A","LOC":"#8AABBA","EVENT":"#C47B6E"
    }
    with driver.session() as session:
        if keyword:
            res = session.run("""
                MATCH (e:Entity) WHERE toLower(e.name) CONTAINS $kw
                WITH e MATCH (e)-[r:CO_OCCURS_WITH]-(other:Entity)
                WITH collect(DISTINCT e)+collect(DISTINCT other) AS nl
                UNWIND nl AS node WITH DISTINCT node
                WITH node, COUNT{(node)--()} AS conn
                ORDER BY conn DESC LIMIT 40
                RETURN node.name AS name, node.type AS type, conn
            """, {"kw": keyword.lower()})
        else:
            res = session.run("""
                MATCH (e:Entity)
                WITH e, COUNT{(e)--()} AS conn
                ORDER BY conn DESC LIMIT 40
                RETURN e.name AS name, e.type AS type, conn
            """)
        top = {r["name"]: r for r in res}
        rels = session.run("""
            MATCH (e1:Entity)-[r:CO_OCCURS_WITH]-(e2:Entity)
            WHERE e1.name IN $names AND e2.name IN $names AND r.count >= 2
            RETURN e1.name AS source, e2.name AS target, r.count AS weight
        """, {"names": list(top.keys())})
        relationships = list(rels)

    net = Network(height="460px", width="100%", bgcolor="#0A0C0F",
                  font_color="#8A9BB0", notebook=False, cdn_resources="in_line")
    net.set_options("""
    {
      "physics": {
        "forceAtlas2Based": {
          "gravitationalConstant": -60,
          "centralGravity": 0.008,
          "springLength": 140,
          "springConstant": 0.06
        },
        "solver": "forceAtlas2Based",
        "stabilization": {"iterations": 120}
      },
      "edges": {
        "smooth": {"type": "continuous"},
        "color": {"opacity": 0.4}
      }
    }
    """)

    for name, data in top.items():
        is_india = name == "India"
        color    = "#E8D5A3" if is_india else TYPE_COLORS.get(data["type"], "#5A6A7A")
        size     = min(12 + data["conn"] * 1.8, 52) if not is_india else 58
        border   = "#FFFFFF" if is_india else color
        net.add_node(
            name, label=name, color={"background": color, "border": border,
            "highlight": {"background": "#E8D5A3", "border": "#FFFFFF"}},
            size=size, title=f"{data['type']} · {data['conn']} connections",
            font={"size": 11, "color": "#C8D4E0", "face": "IBM Plex Mono"}
        )

    for r in relationships:
        if r["source"] in top and r["target"] in top:
            w = min(r["weight"] * 0.4, 4)
            net.add_edge(
                r["source"], r["target"], value=w,
                title=f"co-occurrence: {r['weight']}",
                color={"color": "#2A3A4A", "highlight": "#C8A96E"}
            )

    net.save_graph("graph_visual.html")
    with open("graph_visual.html", "r") as f:
        return f.read()

# ════════════════════════════════════════
# PAGE CONFIG
# ════════════════════════════════════════

st.set_page_config(
    page_title="PRAJNA — Strategic Intelligence",
    page_icon="▪",
    layout="wide",
    initial_sidebar_state="collapsed"
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@300;400;500;600&family=IBM+Plex+Sans:wght@300;400;500;600&display=swap');

*, *::before, *::after { box-sizing: border-box; }

html, body, .stApp {
    background-color: #0A0C0F !important;
    color: #C8D4E0 !important;
    font-family: 'IBM Plex Sans', sans-serif !important;
}

/* Hide Streamlit chrome */
#MainMenu, footer, header { visibility: hidden; }
.stDeployButton { display: none; }
section[data-testid="stSidebar"] { display: none; }

/* Top navigation bar */
.prajna-nav {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 0 20px 0;
    border-bottom: 1px solid #1C2A38;
    margin-bottom: 28px;
}
.prajna-logo {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 13px;
    font-weight: 600;
    letter-spacing: 0.25em;
    color: #E8D5A3;
    text-transform: uppercase;
}
.prajna-logo span {
    color: #4A6A8A;
    margin: 0 12px;
}
.prajna-meta {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px;
    color: #3A5068;
    letter-spacing: 0.1em;
    text-transform: uppercase;
}
.prajna-status {
    display: flex;
    align-items: center;
    gap: 6px;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px;
    color: #4A8A6A;
    letter-spacing: 0.1em;
}
.status-dot {
    width: 6px; height: 6px;
    background: #4A8A6A;
    border-radius: 50%;
    animation: pulse 2s infinite;
}
@keyframes pulse {
    0%, 100% { opacity: 1; }
    50% { opacity: 0.3; }
}

/* Stat row */
.stat-row {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 1px;
    background: #1C2A38;
    border: 1px solid #1C2A38;
    margin-bottom: 28px;
}
.stat-cell {
    background: #0A0C0F;
    padding: 16px 20px;
}
.stat-value {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 22px;
    font-weight: 500;
    color: #E8D5A3;
    line-height: 1;
    margin-bottom: 4px;
}
.stat-label {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 9px;
    color: #3A5068;
    text-transform: uppercase;
    letter-spacing: 0.15em;
}

/* Tabs */
.stTabs [data-baseweb="tab-list"] {
    background: transparent !important;
    border-bottom: 1px solid #1C2A38 !important;
    gap: 0 !important;
    padding: 0 !important;
}
.stTabs [data-baseweb="tab"] {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 10px !important;
    font-weight: 500 !important;
    letter-spacing: 0.15em !important;
    text-transform: uppercase !important;
    color: #3A5068 !important;
    background: transparent !important;
    border: none !important;
    border-bottom: 2px solid transparent !important;
    padding: 10px 20px !important;
    border-radius: 0 !important;
}
.stTabs [aria-selected="true"] {
    color: #E8D5A3 !important;
    border-bottom: 2px solid #E8D5A3 !important;
    background: transparent !important;
}
.stTabs [data-baseweb="tab-panel"] {
    padding-top: 24px !important;
    background: transparent !important;
}

/* Inputs */
.stTextInput > div > div > input {
    background: #0D1117 !important;
    border: 1px solid #1C2A38 !important;
    border-radius: 0 !important;
    color: #C8D4E0 !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 13px !important;
    padding: 10px 14px !important;
    outline: none !important;
}
.stTextInput > div > div > input:focus {
    border-color: #C8A96E !important;
    box-shadow: none !important;
}
.stTextInput label {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 9px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.15em !important;
    color: #3A5068 !important;
}

/* Buttons */
.stButton > button {
    background: transparent !important;
    border: 1px solid #2A3A4A !important;
    border-radius: 0 !important;
    color: #8A9BB0 !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 10px !important;
    font-weight: 500 !important;
    letter-spacing: 0.15em !important;
    text-transform: uppercase !important;
    padding: 8px 20px !important;
    transition: all 0.15s ease !important;
}
.stButton > button:hover {
    background: #1C2A38 !important;
    border-color: #C8A96E !important;
    color: #E8D5A3 !important;
}
.stButton > button:focus {
    box-shadow: none !important;
    outline: none !important;
}

/* Primary action button */
.stButton > button[kind="primary"],
div[data-testid="stButton"] > button:first-child {
    border-color: #C8A96E !important;
    color: #E8D5A3 !important;
}

/* Slider */
.stSlider > div > div > div {
    background: #1C2A38 !important;
}
.stSlider label {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 9px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.15em !important;
    color: #3A5068 !important;
}

/* Brief output box */
.brief-container {
    border: 1px solid #1C2A38;
    border-left: 3px solid #C8A96E;
    background: #0D1117;
    padding: 20px 24px;
    margin: 16px 0;
    font-family: 'IBM Plex Sans', sans-serif;
    font-size: 13px;
    line-height: 1.8;
    color: #A8B8C8;
}
.brief-header {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 9px;
    color: #C8A96E;
    text-transform: uppercase;
    letter-spacing: 0.2em;
    margin-bottom: 14px;
    padding-bottom: 10px;
    border-bottom: 1px solid #1C2A38;
}

/* Path node display */
.path-node {
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 10px 16px;
    margin: 2px 0;
    background: #0D1117;
    border: 1px solid #1C2A38;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 12px;
    color: #C8D4E0;
}
.path-node.start { border-left: 3px solid #6B8CAE; }
.path-node.end   { border-left: 3px solid #C8A96E; }
.path-node.mid   { border-left: 3px solid #2A3A4A; margin-left: 20px; }
.path-connector  {
    margin-left: 28px;
    padding: 4px 16px;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px;
    color: #2A4A3A;
    letter-spacing: 0.1em;
}

/* Source tags */
.source-row {
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
    margin-top: 12px;
    padding-top: 12px;
    border-top: 1px solid #1C2A38;
}
.source-tag {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 9px;
    color: #3A5068;
    border: 1px solid #1C2A38;
    padding: 3px 8px;
    letter-spacing: 0.08em;
    text-transform: uppercase;
}

/* Surge alert card */
.surge-card {
    border: 1px solid #1C2A38;
    border-left: 3px solid #8A4A3A;
    background: #0D1117;
    padding: 16px 20px;
    margin: 8px 0;
}
.surge-entity {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 13px;
    font-weight: 600;
    color: #E8D5A3;
    letter-spacing: 0.1em;
    text-transform: uppercase;
}
.surge-ratio {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 11px;
    color: #C87B6A;
    margin-top: 2px;
}

/* Section labels */
.section-label {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 9px;
    color: #3A5068;
    text-transform: uppercase;
    letter-spacing: 0.2em;
    margin-bottom: 12px;
    padding-bottom: 8px;
    border-bottom: 1px solid #1C2A38;
}

/* Context pills for graph */
.context-item {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 6px 12px;
    border-bottom: 1px solid #0D1117;
    font-family: 'IBM Plex Mono', monospace;
    font-size: 10px;
    color: #5A7A9A;
}
.context-item:hover { background: #0D1117; }
.context-strength {
    color: #3A5068;
    font-size: 9px;
}

/* Metrics override */
div[data-testid="metric-container"] {
    background: #0D1117 !important;
    border: 1px solid #1C2A38 !important;
    border-radius: 0 !important;
    padding: 12px 16px !important;
}
div[data-testid="metric-container"] label {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 9px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.15em !important;
    color: #3A5068 !important;
}
div[data-testid="metric-container"] div[data-testid="stMetricValue"] {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 20px !important;
    color: #E8D5A3 !important;
}

/* Expander */
.streamlit-expanderHeader {
    background: #0D1117 !important;
    border: 1px solid #1C2A38 !important;
    border-radius: 0 !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 10px !important;
    color: #5A7A9A !important;
    letter-spacing: 0.1em !important;
}

/* Bar chart */
div[data-testid="stVegaLiteChart"] { border: 1px solid #1C2A38; }

/* Scrollbar */
::-webkit-scrollbar { width: 4px; }
::-webkit-scrollbar-track { background: #0A0C0F; }
::-webkit-scrollbar-thumb { background: #1C2A38; }

/* Spinner */
.stSpinner > div { border-top-color: #C8A96E !important; }

/* Info / warning boxes */
.stAlert {
    background: #0D1117 !important;
    border: 1px solid #1C2A38 !important;
    border-radius: 0 !important;
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 11px !important;
    color: #5A7A9A !important;
}
</style>
""", unsafe_allow_html=True)

# ════════════════════════════════════════
# NAVIGATION BAR
# ════════════════════════════════════════

now_str = datetime.now(timezone.utc).strftime("%Y-%m-%d  %H:%M UTC")

try:
    articles, entities, rels = get_stats()
    stats_html = f"""
    <div class="stat-row">
        <div class="stat-cell">
            <div class="stat-value">{articles}</div>
            <div class="stat-label">Articles Ingested</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value">{entities}</div>
            <div class="stat-label">Entities Mapped</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value">{rels}</div>
            <div class="stat-label">Relationships</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value" style="color:#4A8A6A">LIVE</div>
            <div class="stat-label">Graph Status</div>
        </div>
    </div>
    """
except:
    stats_html = ""

st.markdown(f"""
<div class="prajna-nav">
    <div>
        <div class="prajna-logo">PRAJNA<span>▪</span>Strategic Intelligence Engine</div>
        <div class="prajna-meta" style="margin-top:4px">
            India Innovates 2026 &nbsp;·&nbsp; Domain 02: Digital Democracy &nbsp;·&nbsp; TeamIIMC
        </div>
    </div>
    <div style="text-align:right">
        <div class="prajna-status">
            <div class="status-dot"></div>
            GRAPH ACTIVE
        </div>
        <div class="prajna-meta" style="margin-top:4px">{now_str}</div>
    </div>
</div>
{stats_html}
""", unsafe_allow_html=True)

# ════════════════════════════════════════
# TABS
# ════════════════════════════════════════

tab1, tab2, tab3, tab4 = st.tabs([
    "INTELLIGENCE BRIEF",
    "TRAJECTORY ANALYSIS",
    "PATH QUERY",
    "SURGE DETECTION"
])

# ── TAB 1: INTELLIGENCE BRIEF ──────────
with tab1:
    col_left, col_right = st.columns([1, 1], gap="large")

    with col_left:
        st.markdown('<div class="section-label">Query Interface</div>', unsafe_allow_html=True)

        query = st.text_input(
            "STRATEGIC QUESTION",
            placeholder="What are India's energy security risks from Iran?",
            key="query_input"
        )

        # Demo query buttons
        st.markdown('<div style="margin: 12px 0 8px; font-family: IBM Plex Mono; font-size: 9px; color: #2A3A4A; letter-spacing: 0.15em; text-transform: uppercase;">Suggested queries</div>', unsafe_allow_html=True)
        demo_queries = [
            "India Iran energy security",
            "Pakistan Afghanistan India impact",
            "India China border tensions",
            "India Russia defense cooperation",
            "Israel Iran war implications",
        ]
        for dq in demo_queries:
            if st.button(dq, key=f"dq_{dq}"):
                st.session_state["query_input"] = dq
                st.rerun()

        st.markdown("<div style='margin-top:16px'></div>", unsafe_allow_html=True)
        run_query = st.button("EXECUTE QUERY ▶", key="run_brief")

        if run_query and query:
            with st.spinner(""):
                keywords     = [w for w in query.lower().split() if len(w) > 3][:5]
                context      = get_graph_context(keywords)
                articles_ctx = get_articles(keywords)

                ctx_str = "\\n".join([f"  {e1} <-> {e2} (strength:{s})"
                                      for e1, e2, s in context])
                art_str = "\\n".join([f"  [{src}] {title}"
                                      for title, src in articles_ctx])

                prompt = f"""You are Prajna, India strategic intelligence engine.
Answer ONLY from the graph context. Cite every claim with source.

KNOWLEDGE GRAPH:
{ctx_str}

NEWS SOURCES:
{art_str}

QUESTION: {query}

Structure response as:
SITUATION — 2-3 sentence summary
KEY CONNECTIONS — 3-4 bullet points from graph
STRATEGIC IMPLICATIONS — what this means for India
RECOMMENDED ACTION — one specific action
SOURCES — list all cited sources"""

                brief = ask_groq(prompt)

            sources_html = ""
            if articles_ctx:
                tags = "".join([f'<span class="source-tag">{src}</span>'
                                for _, src in articles_ctx[:6]])
                sources_html = f'<div class="source-row">{tags}</div>'

            st.markdown(f"""
            <div class="brief-container">
                <div class="brief-header">
                    ▪ INTELLIGENCE BRIEF &nbsp;·&nbsp; {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}
                    &nbsp;·&nbsp; {len(context)} graph connections analysed
                </div>
                {brief.replace(chr(10), "<br>")}
                {sources_html}
            </div>
            """, unsafe_allow_html=True)

        elif run_query and not query:
            st.warning("Enter a query to execute.")

    with col_right:
        st.markdown('<div class="section-label">Knowledge Graph</div>', unsafe_allow_html=True)
        graph_filter = st.text_input(
            "FILTER BY ENTITY",
            placeholder="Iran, China, Taliban...",
            key="graph_filter"
        )
        with st.spinner(""):
            try:
                graph_html = build_graph_visual(graph_filter if graph_filter else None)
                components.html(graph_html, height=460)
                st.markdown("""
                <div style="font-family:IBM Plex Mono;font-size:9px;color:#2A3A4A;
                     text-transform:uppercase;letter-spacing:0.1em;margin-top:8px;">
                    Node size = connection strength &nbsp;·&nbsp;
                    Drag to explore &nbsp;·&nbsp;
                    Hover for details
                </div>
                """, unsafe_allow_html=True)
            except Exception as e:
                st.error(f"Graph render error: {e}")

# ── TAB 2: TRAJECTORY ──────────────────
with tab2:
    st.markdown('<div class="section-label">Relationship Trajectory Analysis</div>', unsafe_allow_html=True)
    st.markdown("""
    <div style="font-family:IBM Plex Mono;font-size:10px;color:#3A5068;margin-bottom:20px;line-height:1.7">
    Tracks how the co-occurrence strength between two entities evolves week-over-week.<br>
    Sustained increases = deepening strategic relationship. Spikes = event-driven. Drops = decoupling.
    </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns([2, 2, 1])
    with col1:
        e1 = st.text_input("ENTITY A", value="India", key="traj_e1")
    with col2:
        e2 = st.text_input("ENTITY B", value="Iran", key="traj_e2")
    with col3:
        st.markdown("<div style='margin-top:28px'></div>", unsafe_allow_html=True)
        run_traj = st.button("RUN ANALYSIS ▶", key="run_traj")

    st.markdown("""
    <div style="font-family:IBM Plex Mono;font-size:9px;color:#2A3A4A;
         letter-spacing:0.1em;margin-bottom:16px">
    SUGGESTED PAIRS — India / Iran &nbsp;·&nbsp; India / Russia &nbsp;·&nbsp;
    Iran / Israel &nbsp;·&nbsp; India / China &nbsp;·&nbsp; Pakistan / Afghanistan
    </div>
    """, unsafe_allow_html=True)

    if run_traj:
        with st.spinner(""):
            weekly_data, total = get_trajectory(e1, e2)

        if not weekly_data:
            st.markdown(f"""
            <div style="font-family:IBM Plex Mono;font-size:11px;color:#5A3A3A;
                 border:1px solid #2A1A1A;padding:14px 18px;background:#0D0A0A">
            ▪ NO TRAJECTORY DATA — {e1.upper()} ↔ {e2.upper()}<br>
            <span style="color:#2A3A4A;font-size:9px">
            These entities may not co-occur in the current graph.
            Try: India/Iran · Iran/Israel · India/Russia
            </span>
            </div>
            """, unsafe_allow_html=True)
        else:
            st.markdown(f"""
            <div style="font-family:IBM Plex Mono;font-size:10px;color:#4A6A4A;
                 border:1px solid #1A2A1A;padding:10px 16px;background:#0A0D0A;
                 margin-bottom:16px">
            ▪ {e1.upper()} ↔ {e2.upper()} &nbsp;·&nbsp;
            {total} TOTAL CO-OCCURRENCES &nbsp;·&nbsp; {len(weekly_data)} WEEK(S) OF DATA
            </div>
            """, unsafe_allow_html=True)

            import pandas as pd
            df = pd.DataFrame(weekly_data, columns=["Week", "Co-occurrences"])
            st.bar_chart(df.set_index("Week"), color="#C8A96E")

            traj_str = "\\n".join([f"  {w}: {c}" for w, c in weekly_data])
            prompt = f"""You are Prajna. Analyse this trajectory for {e1} <-> {e2}:

{traj_str}
Total all-time: {total}

Provide concise analysis:
SITUATION — is this relationship intensifying, cooling, or volatile?
INFLECTION POINTS — any sharp changes and their likely causes
INDIA STRATEGIC IMPLICATIONS — what does this mean for India
WATCH FOR — specific developments to monitor

Be analytical and specific. Max 250 words."""

            with st.spinner(""):
                brief = ask_groq(prompt)

            st.markdown(f"""
            <div class="brief-container">
                <div class="brief-header">▪ TRAJECTORY INTERPRETATION — {e1.upper()} ↔ {e2.upper()}</div>
                {brief.replace(chr(10), "<br>")}
            </div>
            """, unsafe_allow_html=True)

# ── TAB 3: PATH QUERY ──────────────────
with tab3:
    st.markdown('<div class="section-label">Graph Path Query</div>', unsafe_allow_html=True)
    st.markdown("""
    <div style="font-family:IBM Plex Mono;font-size:10px;color:#3A5068;margin-bottom:20px;line-height:1.7">
    Finds the shortest connection path between any two entities in the knowledge graph.<br>
    Surfaces non-obvious relationships invisible to human analysts and impossible for LLMs.
    </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns([2, 2, 1])
    with col1:
        p1 = st.text_input("ORIGIN ENTITY", value="India", key="path_p1")
    with col2:
        p2 = st.text_input("TARGET ENTITY", value="Taliban", key="path_p2")
    with col3:
        st.markdown("<div style='margin-top:28px'></div>", unsafe_allow_html=True)
        run_path = st.button("FIND PATH ▶", key="run_path")

    st.markdown("""
    <div style="font-family:IBM Plex Mono;font-size:9px;color:#2A3A4A;
         letter-spacing:0.1em;margin-bottom:16px">
    SUGGESTED PAIRS — India → Taliban &nbsp;·&nbsp; Russia → Taliban &nbsp;·&nbsp;
    Israel → Pakistan &nbsp;·&nbsp; India → Dubai
    </div>
    """, unsafe_allow_html=True)

    if run_path:
        with st.spinner(""):
            path = find_path(p1, p2)

        if not path:
            st.markdown(f"""
            <div style="font-family:IBM Plex Mono;font-size:11px;color:#5A3A3A;
                 border:1px solid #2A1A1A;padding:14px 18px;background:#0D0A0A">
            ▪ NO PATH FOUND — {p1.upper()} → {p2.upper()}<br>
            <span style="color:#2A3A4A;font-size:9px">
            No connection within 4 hops. Try different entity names or pairs.
            </span>
            </div>
            """, unsafe_allow_html=True)
        else:
            nodes     = path["nodes"]
            strengths = path["strengths"]

            st.markdown(f"""
            <div style="font-family:IBM Plex Mono;font-size:10px;color:#4A6A4A;
                 border:1px solid #1A2A1A;padding:10px 16px;background:#0A0D0A;
                 margin-bottom:16px">
            ▪ PATH FOUND &nbsp;·&nbsp; {path["hops"]} HOP(S) &nbsp;·&nbsp;
            {p1.upper()} → {p2.upper()}
            </div>
            """, unsafe_allow_html=True)

            path_str = ""
            nodes_html = ""
            for i, node in enumerate(nodes):
                if i == 0:
                    nodes_html += f'<div class="path-node start">▶ &nbsp; {node.upper()}</div>'
                    path_str   += node
                elif i == len(nodes)-1:
                    nodes_html += f'<div class="path-node end">◼ &nbsp; {node.upper()}</div>'
                    path_str   += node
                else:
                    nodes_html += f'<div class="path-node mid">◦ &nbsp; {node}</div>'
                    path_str   += node

                if i < len(strengths):
                    nodes_html += f'<div class="path-connector">│ co-occurs {strengths[i]}× in news</div>'
                    path_str   += f" --[{strengths[i]}x]--> "

            st.markdown(nodes_html, unsafe_allow_html=True)

            prompt = f"""You are Prajna. A graph path query revealed this connection:

PATH: {path_str}
HOPS: {path["hops"]}

Each arrow = two entities co-appearing in real news articles.
The number = frequency of co-appearance.

Analyse:
HIDDEN CONNECTION — what does this path mean in plain English?
STRATEGIC SIGNIFICANCE — why does this matter for India specifically?
NON-OBVIOUS INSIGHT — what would an analyst miss without this graph?
RECOMMENDED ACTION — one specific action for Indian policymakers

Be direct and specific. Max 220 words."""

            with st.spinner(""):
                brief = ask_groq(prompt)

            st.markdown(f"""
            <div class="brief-container" style="margin-top:16px">
                <div class="brief-header">▪ PATH INTELLIGENCE — {p1.upper()} → {p2.upper()}</div>
                {brief.replace(chr(10), "<br>")}
            </div>
            """, unsafe_allow_html=True)

# ── TAB 4: SURGE DETECTION ─────────────
with tab4:
    st.markdown('<div class="section-label">Automated Surge Detection</div>', unsafe_allow_html=True)
    st.markdown("""
    <div style="font-family:IBM Plex Mono;font-size:10px;color:#3A5068;margin-bottom:20px;line-height:1.7">
    Monitors the knowledge graph for entities whose news co-occurrence is growing anomalously fast.<br>
    No query required — Prajna surfaces what you did not know to ask about.
    </div>
    """, unsafe_allow_html=True)

    col1, col2 = st.columns([2, 1])
    with col1:
        threshold = st.slider(
            "SURGE THRESHOLD (× WEEKLY BASELINE)",
            1.2, 3.0, 1.5, 0.1
        )
    with col2:
        st.markdown("<div style='margin-top:28px'></div>", unsafe_allow_html=True)
        run_surge = st.button("SCAN GRAPH ▶", key="run_surge")

    if run_surge:
        with st.spinner(""):
            surges = detect_surges(threshold=threshold)

        if not surges:
            st.markdown(f"""
            <div style="font-family:IBM Plex Mono;font-size:11px;color:#3A5068;
                 border:1px solid #1C2A38;padding:14px 18px;background:#0D1117;
                 margin-bottom:16px">
            ▪ NO SURGES DETECTED ABOVE {threshold}× THRESHOLD<br>
            <span style="font-size:9px;color:#2A3A4A">
            Week 1 — baselines being established.
            Surge detection activates fully from Week 2 as historical patterns accumulate.
            </span>
            </div>
            """, unsafe_allow_html=True)

            st.markdown('<div class="section-label" style="margin-top:20px">Current Graph — Top Entities by Connection Strength</div>', unsafe_allow_html=True)
            with driver.session() as session:
                top = session.run("""
                    MATCH (e:Entity)-[r:CO_OCCURS_WITH]-()
                    RETURN e.name AS name, e.type AS type, count(r) AS conn
                    ORDER BY conn DESC LIMIT 12
                """).data()

            rows_html = ""
            for row in top:
                if row["type"] in {"GPE","ORG","PERSON"} and len(row["name"]) < 35:
                    bar_w = min(int(row["conn"] * 3), 200)
                    rows_html += f"""
                    <div class="context-item">
                        <span>{row["name"].upper()}</span>
                        <span style="display:flex;align-items:center;gap:10px">
                            <span style="width:{bar_w}px;height:2px;
                                  background:#1C2A38;display:inline-block"></span>
                            <span class="context-strength">{row["conn"]} conn</span>
                        </span>
                    </div>"""

            st.markdown(f"""
            <div style="border:1px solid #1C2A38;background:#0D1117">
                {rows_html}
            </div>
            """, unsafe_allow_html=True)

        else:
            st.markdown(f"""
            <div style="font-family:IBM Plex Mono;font-size:10px;color:#C87B6A;
                 border:1px solid #2A1A1A;padding:10px 16px;background:#0D0A0A;
                 margin-bottom:16px">
            ▪ {len(surges)} ENTITIES FLAGGED ABOVE {threshold}× BASELINE
            </div>
            """, unsafe_allow_html=True)

            for surge in surges:
                bar_w = min(int(surge["ratio"] * 40), 160)
                st.markdown(f"""
                <div class="surge-card">
                    <div class="surge-entity">▲ {surge["entity"]}</div>
                    <div class="surge-ratio">{surge["ratio"]}× baseline
                        &nbsp;·&nbsp; {surge["latest"]} connections this week
                        &nbsp;·&nbsp; baseline: {surge["baseline"]}/week
                    </div>
                </div>
                """, unsafe_allow_html=True)

                prompt = f"""Prajna surge alert:
Entity: {surge["entity"]} ({surge["type"]})
This week: {surge["latest"]} news connections
Baseline avg: {surge["baseline"]} per week
Surge ratio: {surge["ratio"]}×

SURGE EXPLANATION — why is this entity surging?
INDIA RISK / OPPORTUNITY — strategic implications
URGENCY — Immediate / Days / Weeks
RECOMMENDED ACTION — one specific action

Max 130 words. Be direct."""

                with st.spinner(""):
                    brief = ask_groq(prompt)

                st.markdown(f"""
                <div class="brief-container" style="border-left-color:#8A4A3A;margin-bottom:20px">
                    <div class="brief-header" style="color:#C87B6A">
                        ▪ SURGE BRIEF — {surge["entity"].upper()}
                    </div>
                    {brief.replace(chr(10), "<br>")}
                </div>
                """, unsafe_allow_html=True)
'''

with open("prajna_app.py", "w") as f:
    f.write(app_code)
print("✅ prajna_app.py written — Palantir-style UI ready")

✅ prajna_app.py written — Palantir-style UI ready


In [ ]:
# ============================================
# CELL 9 — Launch Prajna Dashboard
# ============================================
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata
import os

# Set environment variables for the app
os.environ["NEO4J_URI"] = userdata.get("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = userdata.get("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = userdata.get("NEO4J_PASSWORD")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Kill any existing tunnels
ngrok.kill()

# Start Streamlit in background
process = subprocess.Popen([
    "streamlit", "run", "prajna_app.py",
    "--server.port=8501",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false"
])

# Wait for it to start
time.sleep(5)

# Create public URL
tunnel = ngrok.connect(8501)
public_url = tunnel.public_url

print("=" * 60)
print("🚀 PRAJNA IS LIVE!")
print("=" * 60)
print(f"\n🌐 PUBLIC URL: {public_url}")
print(f"\n📋 Share this URL in your PPT!")
print(f"\n⚠️  Keep this Colab tab open — closing it kills the app")
print("=" * 60)

🚀 PRAJNA IS LIVE!

🌐 PUBLIC URL: https://debora-unstanding-adelle.ngrok-free.dev

📋 Share this URL in your PPT!

⚠️  Keep this Colab tab open — closing it kills the app


In [24]:
with open('prajna_app.py', 'r') as f:
    content = f.read()
# Check how spacy is loaded
for i, line in enumerate(content.split('\n')):
    if 'spacy' in line.lower():
        print(f"Line {i}: {line}")

In [25]:
# Read current app
with open('prajna_app.py', 'r') as f:
    content = f.read()

# Replace the stats HTML block with st.metrics instead
old = '''try:
    articles, entities, rels = get_stats()
    stats_html = f"""
    <div class="stat-row">
        <div class="stat-cell">
            <div class="stat-value">{articles}</div>
            <div class="stat-label">Articles Ingested</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value" style="color:#6B8CAE">{entities}</div>
            <div class="stat-label">Entities Mapped</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value" style="color:#C8A96E">{rels}</div>
            <div class="stat-label">Relationships</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value" style="color:#4A8A6A">LIVE</div>
            <div class="stat-label">Graph Status</div>
        </div>
    </div>
    """
except:
    stats_html = ""'''

new = '''try:
    articles, entities, rels = get_stats()
    stats_html = ""
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Articles Ingested", articles)
    c2.metric("Entities Mapped", entities)
    c3.metric("Relationships", rels)
    c4.metric("Graph Status", "LIVE")
except:
    st.warning("Connecting to graph...")
    stats_html = ""'''

content = content.replace(old, new)

# Also remove the stats_html from the nav markdown
old2 = '''{stats_html}
""", unsafe_allow_html=True)'''

new2 = '''""", unsafe_allow_html=True)'''

content = content.replace(old2, new2)

with open('prajna_app.py', 'w') as f:
    f.write(content)
print("✅ Fixed")

✅ Fixed


In [20]:
from google.colab import files
files.download('prajna_app.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
from google.colab import files
uploaded = files.upload()  # upload prajna_logo_v2.svg here

Saving prajna_logo_v2.svg to prajna_logo_v2.svg


In [7]:
import base64

with open('prajna_logo_v2.svg', 'rb') as f:
    logo_b64 = base64.b64encode(f.read()).decode()

# Read current app
with open('prajna_app.py', 'r') as f:
    content = f.read()

# Replace the nav bar section with logo integrated
old = '''st.markdown(f"""
<div class="prajna-nav">
    <div>
        <div class="prajna-logo">PRAJNA<span>▪</span>Strategic Intelligence Engine</div>
        <div class="prajna-meta" style="margin-top:4px">
            India Innovates 2026 &nbsp;·&nbsp; Domain 02: Digital Democracy &nbsp;·&nbsp; TeamIIMC
        </div>
    </div>
    <div style="text-align:right">
        <div class="prajna-status">
            <div class="status-dot"></div>
            GRAPH ACTIVE
        </div>
        <div class="prajna-meta" style="margin-top:4px">{now_str}</div>
    </div>
</div>
{stats_html}
""", unsafe_allow_html=True)'''

new = f'''st.markdown(f"""
<div class="prajna-nav">
    <div style="display:flex;align-items:center;gap:16px">
        <img src="data:image/svg+xml;base64,{logo_b64}"
             width="52" height="52"
             style="opacity:0.95"/>
        <div>
            <div class="prajna-logo">PRAJNA<span>▪</span>Strategic Intelligence Engine</div>
            <div class="prajna-meta" style="margin-top:4px">
                India Innovates 2026 &nbsp;·&nbsp; Domain 02: Digital Democracy &nbsp;·&nbsp; TeamIIMC
            </div>
        </div>
    </div>
    <div style="text-align:right">
        <div class="prajna-status">
            <div class="status-dot"></div>
            GRAPH ACTIVE
        </div>
        <div class="prajna-meta" style="margin-top:4px">{{now_str}}</div>
    </div>
</div>
{{stats_html}}
""", unsafe_allow_html=True)'''

content = content.replace(old, new)

with open('prajna_app.py', 'w') as f:
    f.write(content)

print("✅ Logo integrated into app")

✅ Logo integrated into app


In [12]:
with open('prajna_app.py', 'r') as f:
    content = f.read()

old = '''    articles, entities, rels = get_stats()
    stats_html = f"""
    <div class="stat-row">
        <div class="stat-cell">
            <div class="stat-value">{articles}</div>
            <div class="stat-label">Articles Ingested</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value">{entities}</div>
            <div class="stat-label">Entities Mapped</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value">{rels}</div>
            <div class="stat-label">Relationships</div>
        </div>
        <div class="stat-cell">
            <div class="stat-value" style="color:#4A8A6A">LIVE</div>
            <div class="stat-label">Graph Status</div>
        </div>
    </div>
    """'''

new = '''    articles, entities, rels = get_stats()
    stats_html = ""
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("ARTICLES INGESTED", articles)
    c2.metric("ENTITIES MAPPED", entities)
    c3.metric("RELATIONSHIPS", rels)
    c4.metric("GRAPH STATUS", "● LIVE")'''

if old in content:
    content = content.replace(old, new)
    print("✅ Replaced successfully")
else:
    print("❌ Not found — exact match failed")

# Also remove {stats_html} from the nav render since it's now empty
content = content.replace('\n{stats_html}\n', '\n')

with open('prajna_app.py', 'w') as f:
    f.write(content)
print("✅ File saved")

✅ Replaced successfully
✅ File saved


In [17]:
with open('prajna_app.py', 'r') as f:
    content = f.read()

# Fix 1: Remove duplicate st.rerun()
content = content.replace(
    'st.session_state["query_input"] = dq\n                st.rerun()\n                st.rerun()',
    'st.session_state["query_input"] = dq\n                st.rerun()'
)

# Fix 2: Use a proxy key for session state
content = content.replace(
    '''        query = st.text_input(
            "STRATEGIC QUESTION",
            placeholder="What are India's energy security risks from Iran?",
            key="query_input"
        )''',
    '''        if "query_input" not in st.session_state:
            st.session_state["query_input"] = ""

        query = st.text_input(
            "STRATEGIC QUESTION",
            placeholder="What are India's energy security risks from Iran?",
            value=st.session_state["query_input"],
            key="query_input_box"
        )
        st.session_state["query_input"] = query'''
)

# Fix 3: Update the button to use query_input_box key
content = content.replace(
    'st.session_state["query_input"] = dq\n                st.rerun()',
    'st.session_state["query_input"] = dq\n                st.session_state["query_input_box"] = dq\n                st.rerun()'
)

with open('prajna_app.py', 'w') as f:
    f.write(content)
print("✅ Fixed")

✅ Fixed


In [19]:
with open('prajna_app.py', 'r') as f:
    content = f.read()

# Nuclear fix — replace the entire query input + buttons block
old = '''        if "query_input" not in st.session_state:
            st.session_state["query_input"] = ""

        query = st.text_input(
            "STRATEGIC QUESTION",
            placeholder="What are India's energy security risks from Iran?",
            value=st.session_state["query_input"],
            key="query_input_box"
        )
        st.session_state["query_input"] = query'''

new = '''        if "query_input" not in st.session_state:
            st.session_state["query_input"] = ""

        query = st.text_input(
            "STRATEGIC QUESTION",
            placeholder="What are India's energy security risks from Iran?",
            value=st.session_state["query_input"]
        )'''

content = content.replace(old, new)

# Fix the button — just update session state without widget key conflict
old2 = '''st.session_state["query_input"] = dq\n                st.session_state["query_input_box"] = dq\n                st.rerun()'''
new2 = '''st.session_state["query_input"] = dq\n                st.rerun()'''

content = content.replace(old2, new2)

with open('prajna_app.py', 'w') as f:
    f.write(content)
print("✅ Fixed")

# Verify the query input section looks right
lines = content.split('\n')
for i, line in enumerate(lines[658:688], start=659):
    print(f"{i}: {line}")

✅ Fixed
659:         st.markdown('<div class="section-label">Query Interface</div>', unsafe_allow_html=True)
660: 
661:         if "query_input" not in st.session_state:
662:             st.session_state["query_input"] = ""
663: 
664:         query = st.text_input(
665:             "STRATEGIC QUESTION",
666:             placeholder="What are India's energy security risks from Iran?",
667:             value=st.session_state["query_input"]
668:         )
669: 
670:         # Demo query buttons
671:         st.markdown('<div style="margin: 12px 0 8px; font-family: IBM Plex Mono; font-size: 9px; color: #2A3A4A; letter-spacing: 0.15em; text-transform: uppercase;">Suggested queries</div>', unsafe_allow_html=True)
672:         demo_queries = [
673:             "India Iran energy security",
674:             "Pakistan Afghanistan India impact",
675:             "India China border tensions",
676:             "India Russia defense cooperation",
677:             "Israel Iran war implications",
67